In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm


In [ ]:
# Настройка промпта для California
prompt_config = {
        "task": "Predict whether house price is above median",
        "pos_label": "yes",
        "neg_label": "no",
        "entity": "House",
        "question": "Is this house price above median?"
}

openml_id = 44090

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464):
    dataset = fetch_openml(data_id=openml_id, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name

def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):

    # Разделение на train/val/test (60/20/20)
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df

df, feature_names, target_name = load_dataset(openml_id)
train_df, val_df, test_df = split_dataset(df, target_name)

In [ ]:
df[target_name].value_counts()

,count
price,
0,10317
1,10317


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20634 entries, 0 to 20633
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20634 non-null  float64
 1   HouseAge    20634 non-null  float64
 2   AveRooms    20634 non-null  float64
 3   AveBedrms   20634 non-null  float64
 4   Population  20634 non-null  float64
 5   AveOccup    20634 non-null  float64
 6   Latitude    20634 non-null  float64
 7   Longitude   20634 non-null  float64
 8   price       20634 non-null  int64  
dtypes: float64(8), int64(1)
memory usage: 1.4 MB


In [ ]:
for name, df in [("train", train_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  {prompt_config['neg_label']}: {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  {prompt_config['pos_label']}: {counts[1]:5d} — {pcts[1]:.1f}%")


train (всего: 12380):
  no:  6190 — 50.0%
  yes:  6190 — 50.0%

test (всего: 4127):
  no:  2064 — 50.0%
  yes:  2063 — 50.0%


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names, target_name, prompt_config=None, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    if include_target and prompt_config is not None:
        target_value = prompt_config['pos_label'] if row[target_name] == 1 else prompt_config['neg_label']
        text += f": {target_name} -> {target_value}"

    return text

# Тест
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, True))
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, False))
train_df.head(1)

The value of MedInc is 2.32. The value of HouseAge is 51.00. The value of AveRooms is 3.96. The value of AveBedrms is 1.02. The value of Population is 1173.00. The value of AveOccup is 3.02. The value of Latitude is 34.07. The value of Longitude is -117.76.: price -> no
The value of MedInc is 2.32. The value of HouseAge is 51.00. The value of AveRooms is 3.96. The value of AveBedrms is 1.02. The value of Population is 1173.00. The value of AveOccup is 3.02. The value of Latitude is 34.07. The value of Longitude is -117.76.


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
6451,2.3156,51.0,3.963918,1.015464,1173.0,3.023196,34.07,-117.76,0


In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip()
    response = response.rstrip('.,!? ')

    pos = prompt_config['pos_label'].lower()
    neg = prompt_config['neg_label'].lower()

    # Точное совпадение
    if response == pos:
        return 1
    elif response == neg:
        return 0

    # Начинается с метки
    if response.startswith(pos):
        return 1
    elif response.startswith(neg):
        return 0

    # Содержит метку как отдельное слово
    words = response.split()
    if pos in words:
        return 1
    elif neg in words:
        return 0

    # Не удалось распознать
    print(f"Warning: Could not parse '{response}' (expected '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}')")
    return 0



response = prompt_config['pos_label']
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "hi"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

Response: 'yes'
Prediction: 1

Response: 'hi'
Prediction: 0


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    roc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    pr  = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    return roc, f1, acc, pr, rec


# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name, prompt_config)
print(f"\nПример текста:\n{sample_text[:300]}")


Пример текста:
The value of MedInc is 2.68. The value of HouseAge is 28.00. The value of AveRooms is 4.21. The value of AveBedrms is 1.01. The value of Population is 2218.00. The value of AveOccup is 3.57. The value of Latitude is 34.07. The value of Longitude is -118.18.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            # Пропускание сэмплов, где не представлены все классы
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

# Тестирование функции
sample_row = test_df.iloc[0]
sample_text = row_to_text_template(sample_row, feature_names, target_name, prompt_config)
print(f"\nПример текста:\n{sample_text[:300]}")


Пример текста:
The value of MedInc is 2.68. The value of HouseAge is 28.00. The value of AveRooms is 4.21. The value of AveBedrms is 1.01. The value of Population is 2218.00. The value of AveOccup is 3.57. The value of Latitude is 34.07. The value of Longitude is -118.18.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 3.0 4B Instruct
model_name = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Использовано памяти: 8.04 GB


In [ ]:
def create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples=None):

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"{prompt_config['entity']} information: "
            f"{row_to_text_template(row, feature_names, target_name, prompt_config)}\n"
            f"{prompt_config['question']}"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name, prompt_config)
            ex_target = prompt_config['pos_label'] if ex[target_name] == 1 else prompt_config['neg_label']

            messages.append({
                "role": "user",
                "content": f"{prompt_config['entity']} information: {ex_text}\n{prompt_config['question']}"
            })
            messages.append({
                "role": "assistant",
                "content": ex_target
            })

        client_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        messages.append({
            "role": "user",
            "content": f"{prompt_config['entity']} information: {client_text}\n{prompt_config['question']}"
        })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

def predict_single_with_prob(prompt, prompt_config, model, tokenizer, device, max_new_tokens=5):

    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            attention_mask=model_inputs.attention_mask,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Декодирование текста
    generated_ids = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для меток из prompt_config
    first_token_logits = outputs.scores[0][0]

    pos_id = tokenizer.encode(prompt_config['pos_label'], add_special_tokens=False)[0]
    neg_id = tokenizer.encode(prompt_config['neg_label'], add_special_tokens=False)[0]

    pos_logit = first_token_logits[pos_id]
    neg_logit = first_token_logits[neg_id]

    probs = F.softmax(torch.stack([pos_logit, neg_logit]), dim=0)
    prob_pos = probs[0].item()

    return response, prob_pos

# Тест
test_prompt = create_prompt(test_df.iloc[0], feature_names, target_name, prompt_config, tokenizer)
response, prob = predict_single_with_prob(test_prompt, prompt_config, model, tokenizer, device)
print(f"Response: '{response}', Probability: {prob:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Response: 'no', Probability: 0.3775


# 3.1 Zero-shot

In [ ]:
print("Zero-shot классификация")

seed = 42

# Запускаем zero-shot предсказания
y_true_zero = []
y_pred_zero = []
y_prob_zero = []

start_time = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
    response, prob = predict_single_with_prob(prompt, prompt_config, model, tokenizer, device)
    prediction = parse_prediction(response, prompt_config)

    y_true_zero.append(row[target_name])
    y_pred_zero.append(prediction)
    y_prob_zero.append(prob)

zero_shot_time = time.time() - start_time

print(f"zero_shot_time: {zero_shot_time}")
print(f"zero_shot_time / len(y_true_zero) {zero_shot_time / len(y_true_zero)}")

Zero-shot классификация


  0%|          | 0/4127 [00:00<?, ?it/s]

zero_shot_time: 469.4376950263977
zero_shot_time / len(y_true_zero) 0.11374792707206148


In [ ]:
# Вычисляем метрики
roc_zero, f1_zero, acc_zero, pr_zero, rec_zero = compute_metrics(np.array(y_true_zero), np.array(y_pred_zero), np.array(y_prob_zero))

print("Результаты zero-shot:")
print(f"ROC AUC: {roc_zero}")
print(f"F1 Score: {f1_zero}")
print(f"Accuracy: {acc_zero}")
print(f"Precision: {pr_zero}")
print(f"Recall: {rec_zero}")

Результаты zero-shot:
ROC AUC: 0.7436608038643205
F1 Score: 0.6882295719844358
Accuracy: 0.6893627332202569
Precision: 0.6905807711078575
Recall: 0.6858943286476006


In [ ]:
zero_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_zero),
    np.array(y_pred_zero),
    np.array(y_prob_zero),
    n_iter=1000
)

print("\nРезультаты zero-shot (bootstrap метрики с доверительными интервалами):")
for key, value in zero_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты zero-shot (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.7435±0.0075
  F1: 0.6880±0.0084
  Accuracy: 0.6892±0.0072
  Precision: 0.6900±0.0105
  Recall: 0.6861±0.0101


ROC-AUC (0.74): Модель демонстрирует хорошую разделительную способность даже без обучения. Это указывает на то, что семантика признаков (местоположение, доход, возраст дома) хорошо понятна LLM.

Метрики Precision (0.69) и Recall (0.68) практически идентичны, что говорит о стабильности предсказаний модели в режиме zero-shot для данного набора данных.

Accuracy (0.69): Результат значительно выше случайного угадывания, что подтверждает пригодность модели для анализа социально-экономических табличных данных.

Время тестирования: ~ 8 мин.

# 3.2 Few-shot

In [ ]:
# Выбираем 64 примера из train для контекста
n_examples = 64

seed = 42

# Балансированная выборка
n_per_class = n_examples // 2
pos_examples = train_df[train_df[target_name] == 1].sample(n=n_per_class, random_state=seed)
neg_examples = train_df[train_df[target_name] == 0].sample(n=n_per_class, random_state=seed)
few_shot_df = pd.concat([pos_examples, neg_examples]).sample(frac=1, random_state=seed)

print(f"\nВыбрано {len(few_shot_df)} примеров для контекста:")
print(f"  {prompt_config['pos_label']}: {(few_shot_df[target_name] == 1).sum()}")
print(f"  {prompt_config['neg_label']}: {(few_shot_df[target_name] == 0).sum()}")

few_shot_examples = [row for _, row in few_shot_df.iterrows()]

# Few-shot предсказания
y_true_few = []
y_pred_few = []
y_prob_few = []

start_time = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
    prompt = create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples)
    response, prob = predict_single_with_prob(prompt, prompt_config, model, tokenizer, device)
    prediction = parse_prediction(response, prompt_config)

    y_true_few.append(row[target_name])
    y_pred_few.append(prediction)
    y_prob_few.append(prob)

few_shot_time = time.time() - start_time

print(f"few_shot_time: {few_shot_time}")
print(f"few_shot_time / len(y_true_few) {few_shot_time / len(y_true_few)}")


Выбрано 64 примеров для контекста:
  yes: 32
  no: 32


  0%|          | 0/4127 [00:00<?, ?it/s]

few_shot_time: 2318.582948207855
few_shot_time / len(y_true_few) 0.5618083228029696


In [ ]:
roc_few, f1_few, acc_few, pr_few, rec_few = compute_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few)
)
print("Результаты few-shot:")
print(f"ROC AUC: {roc_few}")
print(f"F1 Score: {f1_few}")
print(f"Accuracy: {acc_few}")
print(f"Precision: {pr_few}")
print(f"Recall: {rec_few}")


Результаты few-shot:
ROC AUC: 0.7443354112885954
F1 Score: 0.7211579395487442
Accuracy: 0.6825781439302157
Precision: 0.6428842504743834
Recall: 0.8211342704798836


In [ ]:
few_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few),
    n_iter=1000
)

print("\nРезультаты few-shot (bootstrap метрики с доверительными интервалами):")
for key, value in few_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")


Результаты few-shot (bootstrap метрики с доверительными интервалами):
  ROC-AUC: 0.7443±0.0076
  F1: 0.7211±0.0076
  Accuracy: 0.6826±0.0073
  Precision: 0.6426±0.0095
  Recall: 0.8214±0.0084


In [ ]:
from google.colab import runtime
runtime.unassign()

ROC-AUC (0.75): Наблюдается небольшой прирост (+0.8%).

Примеры в контексте помогают модели лучше откалибровать решающую границу, но качественного скачка не происходит.

Recall (0.82): Заметный скачок полноты (с 0.68 до 0.82).

Модель стала гораздо лучше идентифицировать целевой класс, но это произошло за счет значительного снижения точности.

Precision (0.64) и Accuracy (0.68): Снижение точности при росте Recall указывает на смещение (bias) модели.

Под влиянием примеров модель стала чаще предсказывать положительный класс, что увеличило число ложноположительных срабатываний.

Время выполнения: ~ 38 мин.

Использование оперативная памяти графического процесса: 10.4/80GB.

Графический процессор A100 80GB.